In [1]:
import sys
import numpy as np

sys.path.append('../../../src/')
from Rain.Rain import Rain
sys.path.pop()

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "cloud",
      "params": {
        "num_of_workers": 1,
          "subscription_id":'6e14c264-a7fc-4db4-a23a-d972c21a2d99', # Menna's ID
          "location": 'eastus'
      }
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 2,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 16,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/breast_cancer/train_data.npy"), np.load(
        "../../../data/breast_cancer/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/breast_cancer/test_data.npy"), np.load(
        "../../../data/breast_cancer/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 30
    num_labels = 2
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train = np.reshape(X_train, [-1, 30])
y_train = to_categorical(y_train)

In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-06 19:17:24,711 [DEBUG] [Rain] Rain is initialized
2023-07-06 19:17:24,716 [DEBUG] [Provisioner] Creating coordinator
2023-07-06 19:17:24,717 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\coord/
2023-07-06 19:17:24,719 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-06 19:17:24,726 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\prov/
2023-07-06 19:17:24,728 [DEBUG] [CloudProvisioner] Provisioner is initialized
2023-07-06 19:17:24,732 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-06 19:17:24,735 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-06 19:17:24,737 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/


In [8]:
# model = rain.train(X_train, y_train, strategy='async')

In [9]:
# X_test, y_test = get_test_data()
# X_test = np.reshape(X_test, [-1, 30])
# y_test = to_categorical(y_test)
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-06 19:17:25,040 [INFO] [Provisioner] provisioner is serving
2023-07-06 19:17:25,042 [DEBUG] [Provisioner] Starting coordinator
2023-07-06 19:17:25,047 [INFO] [Coordinator] coordinator is serving
2023-07-06 19:17:25,048 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-06 19:17:25,063 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-06 19:17:25,067 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-06 19:17:25,069 [DEBUG] [CloudProvisioner] Setup the networking for the workers
2023-07-06 19:17:28,861 [DEBUG] [CloudProvisioner] Created resource group: Rain-resourcegroup
2023-07-06 19:17:43,188 [DEBUG] [CloudProvisioner] Created virtual network: Rain-vnet
2023-07-06 19:17:45,626 [DEBUG] [CloudProvisioner] Created network security group: Rain-nic-nsg
2023-07-06 19:17:45,626 [DEBUG] [CloudProvisioner] Network setup completed
2023-07-06 19:17:45,627 [DEBU

In [11]:
X_test, y_test = get_test_data()
X_test = np.reshape(X_test, [-1, 30])
y_test = to_categorical(y_test)
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.